# Training Pyxie: Self-Play with MaskablePPO

This notebook shows the minimal setup we used to train the **Pyxie** reinforcement learning agent via self-play. Pyxie is trained using [MaskablePPO](https://sb3-contrib.readthedocs.io/en/master/modules/ppo_mask.html) from `sb3-contrib`, where the agent plays against frozen copies of itself that are periodically updated with the latest policy weights.

**Prerequisites:**
```bash
uv sync
pip install -r notebooks/requirements.txt
```

SB3, sb3-contrib, PyTorch, and Gymnasium are already core dependencies of the package.

In [ ]:
import functools
from typing import Optional

import torch
from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.callbacks import MaskableEvalCallback
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor, VecNormalize

from pyxis_portfolio_challenge.environment import (
    SelfPlayWrapper,
    make_multi_agent_train_env,
)
from pyxis_portfolio_challenge.environment.self_play import OpponentSyncCallback

## Hyperparameters

These are the values we used to train the shipped Pyxie model. Key choices:

- **50 parallel envs** for throughput — each subprocess holds its own opponent policy copy
- **Large network** (1024 → 512 → 256) to handle the high-dimensional observation space
- **gamma=0.98** for long-horizon drug development payoffs
- **Flat learning rate** at 1e-5 — we found a constant low rate worked well for fine-tuning via self-play
- **gae_lambda=0.80** to account for the high stochasticity in the environment (lower than the typical 0.95 default to reduce variance in advantage estimates)
- **target_kl=0.03** for early stopping within each epoch to prevent large policy updates

In [ ]:
# Environment
N_ENVS = 50
N_EVAL_ENVS = 10
TOTAL_TIMESTEPS = 200_000_000

# Policy architecture
POLICY_KWARGS = {"net_arch": [1024, 512, 256]}

# Learning rate (constant)
LEARNING_RATE = 1e-5

# Evaluation
EVAL_FREQ = 5000
N_EVAL_EPISODES = 50

## Environment Setup

`make_multi_agent_train_env()` creates the PettingZoo multi-agent environment from the YAML config. `SelfPlayWrapper` converts it into a single-agent `gym.Env` that SB3 can train on — it combines investments and BD bids into a single `MultiDiscrete` action space, and holds a frozen opponent policy copy that gets periodically synced with the latest training weights.

We wrap in `SubprocVecEnv` for parallel rollout collection, `VecMonitor` for episode tracking, and `VecNormalize` for observation and reward normalization.

### Reward Shaping

The environment emits a raw net cash flow reward each step (cash after − cash before). These values span a huge range (e.g. ±10 billion) which makes learning unstable even with `VecNormalize`. We apply **symmetric log compression** — `sign(x) · log(1 + |x|)` — to squash the reward into a ~±25 range while preserving sign and ordering. This is implemented as a `gym.RewardWrapper` so it sits between the environment and the normalizer.

In [ ]:
import math

import gymnasium as gym


class SymLogRewardWrapper(gym.RewardWrapper):
    """Apply symmetric log compression to rewards: sign(r) * log(1 + |r|)."""

    def reward(self, reward):  # noqa: D102
        return math.copysign(math.log1p(abs(reward)), reward)


def make_env(seed):  # noqa: D103
    def _init():
        env = make_multi_agent_train_env()
        wrapped = SelfPlayWrapper(env, policy_kwargs=POLICY_KWARGS)
        wrapped = SymLogRewardWrapper(wrapped)
        wrapped.reset(seed=seed)
        return wrapped
    return _init


# Training environment
train_env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)])
train_env = VecMonitor(train_env)
train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True)

In [ ]:
# Evaluation environment (separate, for tracking best model during training)
eval_env = SubprocVecEnv([make_env(1000 + i) for i in range(N_EVAL_ENVS)])
eval_env = VecMonitor(eval_env)
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=True)

## Model

We use `MaskablePPO` from sb3-contrib, which extends PPO with action masking support. The environment's `action_masks()` method tells the policy which actions are valid at each step (e.g. only idle assets can receive new investments), preventing the agent from wasting capacity learning to avoid invalid actions.

In [ ]:
model = MaskablePPO(
    "MlpPolicy",
    train_env,
    policy_kwargs=POLICY_KWARGS,
    gamma=0.98,
    ent_coef=0.02,
    learning_rate=LEARNING_RATE,
    clip_range=0.1,
    n_steps=8192,
    batch_size=512,
    n_epochs=5,
    target_kl=0.03,
    gae_lambda=0.80,
    verbose=1,
)

## Weighted Entropy

The action space has two components: investment decisions (high impact) and BD auction bids (lower impact). By default, PPO's entropy bonus weights all action dimensions equally, which wastes exploration budget on BD bids. We patch the policy to apply per-dimension entropy weights: 1.0 for investment dimensions, 0.1 for BD bid dimensions.

In [ ]:
def make_entropy_weights(action_space, bd_slots=3, bd_entropy_weight=0.1):
    """
    Create per-dimension entropy weights.

    Investment dimensions get weight 1.0, BD bid dimensions get bd_entropy_weight.
    """
    n_dims = len(action_space.nvec)
    weights = torch.ones(n_dims)
    if bd_slots > 0 and bd_slots <= n_dims:
        weights[-bd_slots:] = bd_entropy_weight
    return weights


def patch_policy_entropy_weights(model, weights):
    """
    Monkey-patch MaskablePPO to use per-dimension entropy weights.

    Wraps evaluate_actions so entropy is a weighted sum across action dimensions
    instead of a uniform mean.
    """
    policy = model.policy
    weights = weights.to(policy.device)
    norm_weights = weights / weights.sum()

    original_evaluate_actions = policy.evaluate_actions

    @functools.wraps(original_evaluate_actions)
    def weighted_evaluate_actions(
        obs: torch.Tensor,
        actions: torch.Tensor,
        action_masks: Optional[torch.Tensor] = None,
    ):
        values, log_prob, entropy = original_evaluate_actions(
            obs, actions, action_masks=action_masks
        )
        dist = policy.action_dist
        has_dists = hasattr(dist, "distributions")
        if has_dists and len(dist.distributions) == len(norm_weights):
            per_dim_entropy = torch.stack(
                [d.entropy() for d in dist.distributions], dim=-1
            )
            entropy = (per_dim_entropy * norm_weights.unsqueeze(0)).sum(dim=-1)
        return values, log_prob, entropy

    policy.evaluate_actions = weighted_evaluate_actions


# Apply weighted entropy
weights = make_entropy_weights(model.action_space, bd_slots=3, bd_entropy_weight=0.1)
patch_policy_entropy_weights(model, weights)

## Callbacks

Three callbacks coordinate the training loop:

- **`OpponentSyncCallback`**: At the start of each rollout, pushes the latest policy weights and observation normalization stats to all opponent copies across subprocess environments. `sync_every_n_rollouts=1` means opponents are always up to date.
- **`MaskableEvalCallback`**: Periodically evaluates the policy against the current opponent and saves the best model.
- **`CheckpointCallback`**: Saves periodic snapshots (model + VecNormalize stats) for recovery.

In [ ]:
opponent_sync = OpponentSyncCallback(sync_every_n_rollouts=1, eval_env=eval_env)

eval_callback = MaskableEvalCallback(
    eval_env,
    best_model_save_path="./best_model/",
    log_path="./logs/",
    eval_freq=EVAL_FREQ,
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=True,
)

checkpoint_callback = CheckpointCallback(
    save_freq=250_000,
    save_path="./checkpoints/",
    save_vecnormalize=True,
)

callbacks = [opponent_sync, eval_callback, checkpoint_callback]

## Training

We trained Pyxie for 200M timesteps. This takes a significant amount of time — reduce `TOTAL_TIMESTEPS` for experimentation (e.g. 1M for a quick test run).

In [ ]:
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callbacks)

## Save Model and Normalizer

Both the model weights **and** the `VecNormalize` running statistics must be saved together. The normalizer is required at inference time — without it, observations will be on a different scale and the policy will produce garbage actions.

In [ ]:
model.save("pyxie_model.zip")
train_env.save("vecnormalize.pkl")

## Next Steps

- **Evaluate** your trained model against other agents using the competition API:
  ```python
  from pyxis_portfolio_challenge.environment.competition import evaluate
  
  per_agent_reports, global_report, playthrough = evaluate(
      agents=[my_agent, "knapsack(c12)"],
      num_episodes=100,
      num_workers=4,
  )
  ```
- **Load your model** for inference — see `MultiAgentPyxieAgent` in `pyxis_portfolio_challenge/agents/multi_agent_pyxie.py` for how to wrap a saved MaskablePPO model as a callable agent.
- **TensorBoard**: pass `tensorboard_log="./tb_logs/"` to `MaskablePPO` to enable SB3's built-in TensorBoard logging.